# 🏆 Workshop: Bau deinen eigenen WM-Experten 2026
## Notebook 1: Verbindung zum MCP-Server & Tool-Aufrufe

In diesem Notebook lernen wir das **Model Context Protocol (MCP)** kennen. MCP ist ein offenes Protokoll, das es KIs (wie ChatGPT, Claude oder euren eigenen Agenten) ermöglicht, auf externe Datenquellen und Tools zuzugreifen.

Unser WM-Experten-Agent benötigt Zugriff auf aktuelle Spieldaten, ELO-Ratings und Simulations-Algorithmen. Diese werden von unserem maßgeschneiderten **MCP-Server** bereitgestellt.

### Lernziele:
1. Verstehen, wie ein MCP-Client mit einem MCP-Server über HTTP kommuniziert.
2. Abfragen der verfügbaren Tools des MCP-Servers.
3. Ausführen von Tools wie ELO-Abfragen und der Turnier-Simulation über Python.


### Schritt 1: Verbindung testen und Tools auflisten
Unser MCP-Server läuft als FastAPI-Anwendung und stellt eine REST-API bereit. Der standardmäßige Endpunkt `/tools` listet alle Funktionen auf, die die KI aufrufen kann. Wir nutzen die Python-Bibliothek `httpx`, um diesen aufzurufen.


In [ ]:
import httpx
import json

# Die URL des MCP-Servers
MCP_URL = "https://wm2026.fh-swf.cloud/mcp"

try:
    response = httpx.get(f"{MCP_URL}/tools")
    response.raise_for_status()
    data = response.json()
    tools = data.get("tools", [])
    
    print(ff"✅ Erfolgreich verbunden! {len(tools)} Tools gefunden:\n")
    for tool in tools:
        print(ff"🛠️  Tool-Name: {tool['name']}")
        print(ff"   Beschreibung: {tool['description']}")
        print("   Parameter:")
        print(json.dumps(tool["parameters"], indent=4, ensure_ascii=False))
        print("-" * 60)
except Exception as e:
    print(f"❌ Fehler bei der Verbindung zum MCP-Server: {e}")
    print("HINWEIS: Bitte stelle sicher, dass der MCP-Server läuft! Startbefehl im Terminal: uv run wm mcp")


### Schritt 2: ELO-Statistiken abfragen
Wir rufen nun das Tool `get_team_elo` auf. Dazu senden wir einen `POST`-Request an den `/call`-Endpunkt des MCP-Servers mit dem gewünschten Tool-Namen und den Parametern.


In [ ]:
def call_tool(name, parameters={}):
    """Hilfsfunktion, um ein MCP-Tool aufzurufen."""
    response = httpx.post(f"{MCP_URL}/call", json={"tool": name, "parameters": parameters})
    response.raise_for_status()
    return response.json().get("result", {})

# ELO-Daten für Deutschland abfragen
deutschland_elo = call_tool("get_team_elo", {"team": "Deutschland"})
print(json.dumps(deutschland_elo, indent=2, ensure_ascii=False))


### Schritt 3: Turniersimulation ausführen
Der MCP-Server verfügt über eine Monte-Carlo-Simulation für das gesamte Turnier (`simulate_tournament`). Wir rufen das Tool auf und lassen die Chancen berechnen.


In [ ]:
# Simulation starten
sim_result = call_tool("simulate_tournament", {"team": "Deutschland", "n_sims": 1000})

print("=== WM 2026 Prognose für Deutschland ===")
print(sim_result.get("summary"))
print("\nVisualisierung der Wahrscheinlichkeiten pro Runde:")
print(sim_result.get("bar_chart"))


### 🧠 Aufgabe für dich:
Schreibe ein kleines Skript, das die Siegwahrscheinlichkeiten der Top-10-Teams abfragt und ausgibt. Nutze dazu das Tool `simulate_tournament` ohne Angabe eines Teams.


In [ ]:
# DEINE LÖSUNG HIER:
top10_result = call_tool("simulate_tournament", {"n_sims": 1000})
print(top10_result.get("bar_chart"))
